# 2.0 Feature Exploration — Random Split A

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from itertools import combinations

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.fully_connected import train_model
from src.helper import make_run_dir, save_run

## Load data

In [ ]:
DATA_SET = 'rand_A'

df_train = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_train.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_val.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_test.parquet'))

OUTPUT_DIR = make_run_dir()

## Hyperparameters

In [ ]:
BATCH_SIZE    = 4096
MAX_EPOCHS    = 50
PATIENCE      = 30
LR_PATIENCE   = 8
LR_FACTOR     = 0.3
INIT_LR       = 1e-3
ACTIVATION    = 'relu'
NEURONS       = 80
HIDDEN_LAYERS = 3

## Feature definitions

In [ ]:
FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'iv_lag', 'd_iv_lag','gamma', 'theta', 'vega','vix_mom_lag', 'vix_mom', 'rho']
TARGET        = 'd_iv'

train_cfg = dict(
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, lr=INIT_LR,
    patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
    hidden_layers=HIDDEN_LAYERS, neurons=NEURONS, activation=ACTIVATION,
    target=TARGET,
)


# Build all combos: base, +1, +2, +3
feature_combos = []

# Base 3F
feature_combos.append(('3F', FEATURES_3))

# 3F + 1
for combo in combinations(EXTRA_FEATURES, 1):
    name = '3F+' + '+'.join(combo)
    feature_combos.append((name, FEATURES_3 + list(combo)))

# 3F + 2
for combo in combinations(EXTRA_FEATURES, 2):
    name = '3F+' + '+'.join(combo)
    feature_combos.append((name, FEATURES_3 + list(combo)))

# 3F + 3
for combo in combinations(EXTRA_FEATURES, 3):
    name = '3F+' + '+'.join(combo)
    feature_combos.append((name, FEATURES_3 + list(combo)))

print(f'Total feature combinations: {len(feature_combos)}')
print(f'  Base (3F):  1')
print(f'  3F + 1:     {len(list(combinations(EXTRA_FEATURES, 1)))}')
print(f'  3F + 2:     {len(list(combinations(EXTRA_FEATURES, 2)))}')
print(f'  3F + 3:     {len(list(combinations(EXTRA_FEATURES, 3)))}')

## Analytic benchmark

In [ ]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

## Train all feature combinations

In [ ]:
all_results = {}

for i, (fname, feats) in enumerate(feature_combos):
    print(f'\n=== [{i+1}/{len(feature_combos)}] {fname} ({len(feats)}F) ===')
    print(f'    Features: {feats}')

    try:
        result = train_model(
            df_train, df_val, df_test,
            features=feats,
            desc=fname,
            **train_cfg,
        )
        all_results[fname] = result

        g = (1 - result['sse'] / hw['sse']) * 100
        print(f'    Gain vs HW: {g:.2f}%')

    except Exception as e:
        print(f'    FAILED: {e}')

print(f'\n\nDone - {len(all_results)}/{len(feature_combos)} models trained.')

## Results summary

In [ ]:
summary = save_run(
    run_dir=OUTPUT_DIR,
    y_test=df_test[TARGET].values,
    hw=hw,
    models=all_results,
)

rows = []
for name, result in all_results.items():
    feats = [f for f in result['features']] if 'features' in result else [name]
    rows.append({
        'combo_name': name,
        'n_features': len([c for c in feature_combos if c[0] == name][0][1]),
        'features': ', '.join([c for c in feature_combos if c[0] == name][0][1]),
        'SSE': result['sse'],
        'RMSE': result['rmse'],
        'Gain_vs_HW_%': (1 - result['sse'] / hw['sse']) * 100,
        'training_time_s': result['training_time'],
        'epochs_run': len(result['history']['loss']),
    })

df_results = pd.DataFrame(rows).sort_values('SSE').reset_index(drop=True)
df_results.to_csv(OUTPUT_DIR / 'feature_exploration_results.csv', index=False)

summary

## Top 10 by Gain vs Hull-White

In [ ]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

## Best per group (3F, +1, +2, +3)

In [ ]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f'{nf_label}: {best["combo_name"]}')
    print(f'    SSE={best["SSE"]:.4f}  RMSE={best["RMSE"]:.6f}  Gain={best["Gain_vs_HW_%"]:.2f}%\n')

## Summary statistics

In [ ]:
total_time = df_results['training_time_s'].sum()
print(f'Total training time: {total_time/60:.1f} min ({total_time/3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')